# TDI Python 2.0 | Week 4: File Handling & Data Quality Assessment

**Project:** Supply Chain Data Quality Assessment  
**Source:** 3,000-row supply chain dataset  
**Tools:** Python standard library (`csv`, `datetime`, `collections`)

## Load the dataset and define cleaning helpers

This section reads the raw CSV and defines reusable functions for dates, suppliers, warehouses and delivery statuses. Each helper is documented in the code.


In [1]:
# Import standard-library tools. No third-party packages are required.
import csv
import collections
import datetime
from pathlib import Path

# Look for the original dataset in the same folder as this notebook.
# The second filename covers the name GitHub/Windows may give a duplicate download.
data_folder = Path('.')
possible_files = [data_folder / 'supply_chain_dataset.csv',
                  data_folder / 'supply_chain_dataset (1).csv']
source_file = next((file for file in possible_files if file.exists()), None)
if source_file is None:
    raise FileNotFoundError('Place supply_chain_dataset.csv beside this notebook.')

# DictReader represents each CSV record as a dictionary keyed by column name.
with source_file.open(newline='', encoding='utf-8-sig') as file:
    reader = csv.DictReader(file)
    columns = reader.fieldnames
    rows = list(reader)

print(f'Loaded {len(rows):,} orders from {source_file.name}')
print('Columns:', columns)

# Convert either MM/DD/YYYY or YYYY-MM-DD into a Python date object.
def parse_date(value):
    if not value.strip():
        return None
    date_format = '%m/%d/%Y' if '/' in value else '%Y-%m-%d'
    return datetime.datetime.strptime(value, date_format).date()

# Treat variants such as ' Supplier A ' and 'suppliera' as SupplierA.
def clean_supplier(value):
    compact = value.strip().replace(' ', '').lower()
    if compact.startswith('supplier') and compact[-1:] in 'abcde':
        return 'Supplier' + compact[-1].upper()
    return compact

# Expand warehouse abbreviations while leaving other labels unchanged.
def clean_warehouse(value):
    value = value.strip()
    return 'Warehouse_' + value[3:] if value.startswith('WH_') else value

# Correct only the two known delivery-status spelling mistakes.
def clean_status(value):
    value = value.strip()
    corrections = {'Delievered': 'Delivered', 'Pendng': 'Pending'}
    return corrections.get(value, value)

# Create a NEW cleaned record; do not overwrite the original dataset.
def standardise_record(row):
    cleaned = {column: value.strip() for column, value in row.items()}
    cleaned['supplier_name'] = clean_supplier(cleaned['supplier_name'])
    cleaned['warehouse_destination'] = clean_warehouse(cleaned['warehouse_destination'])
    cleaned['delivery_status'] = clean_status(cleaned['delivery_status'])

    # Convert valid dates to ISO YYYY-MM-DD; leave invalid values for review.
    for column in ('order_date', 'delivery_date'):
        if cleaned[column]:
            try:
                cleaned[column] = parse_date(cleaned[column]).isoformat()
            except ValueError:
                pass
    return cleaned


Loaded 3,000 orders from supply_chain_dataset (1).csv
Columns: ['order_id', 'order_date', 'product_category', 'quantity', 'unit_price', 'total_cost', 'supplier_name', 'supplier_location', 'lead_time_days', 'warehouse_destination', 'delivery_date', 'actual_delivery_days', 'delivery_status', 'on_time', 'product_defects', 'return_status']


## Task 1: Inspect missing values, duplicates and inconsistent formats

We inspect the **original** records before making any changes.


In [2]:
# Count blank cells separately for every column.
missing_counts = {
    column: sum(not row[column].strip() for row in rows)
    for column in columns
}

# A tuple of all column values can be stored in a set to spot exact repeats.
seen_records = set()
duplicate_order_ids = []
for row in rows:
    record_key = tuple(row[column] for column in columns)
    if record_key in seen_records:
        duplicate_order_ids.append(row['order_id'])
    seen_records.add(record_key)

print('Missing cells:', missing_counts)
print('Exact duplicate rows:', len(duplicate_order_ids))

# Compare unique raw labels to reveal inconsistent spelling or spacing.
for column in ('supplier_name', 'warehouse_destination',
               'delivery_status', 'product_category'):
    unique_values = sorted({row[column] for row in rows})
    print(f'{column}: {len(unique_values)} distinct values:', unique_values)

# Check whether category labels contain unwanted surrounding whitespace.
category_whitespace = sum(
    row['product_category'] != row['product_category'].strip()
    for row in rows
)
print('Category cells with surrounding spaces:', category_whitespace)

# Summarise date formatting without changing the original data.
for column in ('order_date', 'delivery_date'):
    date_formats = collections.Counter(
        'blank' if not row[column].strip()
        else ('slash' if '/' in row[column] else 'ISO')
        for row in rows
    )
    print(column, dict(date_formats))


Missing cells: {'order_id': 0, 'order_date': 0, 'product_category': 0, 'quantity': 0, 'unit_price': 0, 'total_cost': 0, 'supplier_name': 0, 'supplier_location': 0, 'lead_time_days': 0, 'warehouse_destination': 0, 'delivery_date': 0, 'actual_delivery_days': 0, 'delivery_status': 0, 'on_time': 0, 'product_defects': 154, 'return_status': 146}
Exact duplicate rows: 0
supplier_name: 15 distinct values: ['Supplier A', 'Supplier B', 'Supplier C', 'Supplier D', 'Supplier E', 'SupplierA', 'SupplierB', 'SupplierC', 'SupplierD', 'SupplierE', 'suppliera', 'supplierb', 'supplierc', 'supplierd', 'suppliere']
warehouse_destination: 12 distinct values: ['WH_1', 'WH_2', 'WH_3', 'WH_4', 'WH_5', 'WH_6', 'Warehouse_1', 'Warehouse_2', 'Warehouse_3', 'Warehouse_4', 'Warehouse_5', 'Warehouse_6']
delivery_status: 6 distinct values: ['Cancelled', 'Delayed', 'Delievered', 'Delivered', 'In-Transit', 'Pendng']
product_category: 8 distinct values: ['Connectors', 'Control Systems', 'Fasteners', 'Hydraulics', 'Motor

## Task 2: Check logical consistency

The validation function returns issue labels for each order. One order may have multiple issues.


In [3]:
# Non-final deliveries may legitimately lack an actual delivery date/duration.
non_final_statuses = {'Cancelled', 'In-Transit', 'Pending', 'Pendng'}

def validate_order(row):
    """Return a list of quality problems found in one original order."""
    issues = []
    delivery_status = clean_status(row['delivery_status'])

    # Flag missing values, except expected delivery blanks for non-final orders.
    for column in columns:
        is_expected_blank = (
            column in ('delivery_date', 'actual_delivery_days')
            and delivery_status in non_final_statuses
        )
        if not row[column].strip() and not is_expected_blank:
            issues.append('missing_' + column)

    # Compare recorded cost with quantity multiplied by unit price.
    # A difference above 1 unit is treated as a mismatch (same Week 4 rule).
    try:
        cost = float(row['total_cost'])
        quantity = float(row['quantity'])
        unit_price = float(row['unit_price'])
        if cost <= 0:
            issues.append('nonpositive_cost')
        if abs(cost - quantity * unit_price) > 1:
            issues.append('cost_mismatch')
    except (ValueError, TypeError):
        issues.append('invalid_cost_or_inputs')

    # Completed deliveries should not precede the order date.
    if delivery_status not in non_final_statuses:
        try:
            order_date = parse_date(row['order_date'])
            delivery_date = parse_date(row['delivery_date'])
            if order_date and delivery_date:
                calculated_days = (delivery_date - order_date).days
                if calculated_days < 0:
                    issues.append('delivery_before_order')
                try:
                    if int(row['actual_delivery_days']) != calculated_days:
                        issues.append('duration_mismatch')
                except ValueError:
                    issues.append('invalid_actual_days')
        except ValueError:
            issues.append('invalid_delivery_date')

        # The on-time flag should agree with actual days <= lead-time days.
        try:
            actual_days = int(row['actual_delivery_days'])
            lead_days = int(row['lead_time_days'])
            flag = row['on_time'].strip().lower()
            if flag in ('yes', 'no') and (flag == 'yes') != (actual_days <= lead_days):
                issues.append('on_time_mismatch')
        except ValueError:
            issues.append('invalid_lead_or_actual')
    return issues

# Run validation for every record, keeping issues aligned with original rows.
all_issues = [validate_order(row) for row in rows]
issue_counts = collections.Counter(
    issue for order_issues in all_issues for issue in order_issues
)
non_final_count = sum(
    clean_status(row['delivery_status']) in non_final_statuses for row in rows
)
print('Non-final orders:', non_final_count)
for issue, count in sorted(issue_counts.items()):
    # Show a few order IDs so problems can be investigated later.
    example_ids = [row['order_id'] for row, problems in zip(rows, all_issues)
                   if issue in problems][:5]
    print(issue, count, 'examples:', example_ids)


Non-final orders: 107
cost_mismatch 154 examples: ['ORD-0029', 'ORD-0034', 'ORD-0057', 'ORD-0085', 'ORD-0096']
delivery_before_order 23 examples: ['ORD-0007', 'ORD-0060', 'ORD-0122', 'ORD-0331', 'ORD-0408']
duration_mismatch 23 examples: ['ORD-0007', 'ORD-0060', 'ORD-0122', 'ORD-0331', 'ORD-0408']
missing_product_defects 154 examples: ['ORD-0010', 'ORD-0036', 'ORD-0050', 'ORD-0052', 'ORD-0064']
missing_return_status 146 examples: ['ORD-0003', 'ORD-0030', 'ORD-0052', 'ORD-0055', 'ORD-0058']
nonpositive_cost 1 examples: ['ORD-1521']
on_time_mismatch 54 examples: ['ORD-0033', 'ORD-0039', 'ORD-0081', 'ORD-0184', 'ORD-0211']


## Task 3: Standardise and export

Standardise known text variants and valid dates, then verify that the exported CSV contains the same number of rows. Unresolved business-rule problems remain flagged, not silently changed.


In [4]:
# Build cleaned records and count changed cells for each column.
change_counts = collections.Counter()
standardised_rows = []
for original_row in rows:
    cleaned_row = standardise_record(original_row)
    standardised_rows.append(cleaned_row)
    for column in columns:
        if cleaned_row[column] != original_row[column]:
            change_counts[column] += 1

# Export the cleaned records to a NEW file with the original column order.
output_file = data_folder / 'supply_chain_standardised.csv'
with output_file.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    writer.writerows(standardised_rows)

# Read the exported file back to verify that no records were lost.
with output_file.open(newline='', encoding='utf-8') as file:
    verified_rows = list(csv.DictReader(file))
assert len(verified_rows) == len(rows), 'Exported row count does not match!'

print('Saved:', output_file)
print('Verified rows:', len(verified_rows))
print('Changes by column:', dict(change_counts))
for column in ('supplier_name', 'warehouse_destination', 'delivery_status'):
    print(column, 'after:', sorted({row[column] for row in verified_rows}))


Saved: supply_chain_standardised.csv
Verified rows: 3000
Changes by column: {'order_date': 3000, 'delivery_date': 3000, 'supplier_name': 1945, 'warehouse_destination': 276, 'delivery_status': 43}
supplier_name after: ['SupplierA', 'SupplierB', 'SupplierC', 'SupplierD', 'SupplierE']
warehouse_destination after: ['Warehouse_1', 'Warehouse_2', 'Warehouse_3', 'Warehouse_4', 'Warehouse_5', 'Warehouse_6']
delivery_status after: ['Cancelled', 'Delayed', 'Delivered', 'In-Transit', 'Pending']


## Extra challenges: Flagged orders by supplier and month

Count records with at least one issue and group them using dictionaries and `Counter`.


In [5]:
# Challenge 1: Count ORDERS with at least one issue, not total issue flags.
problem_order_count = sum(bool(problems) for problems in all_issues)
print('Challenge 1: rows failing validate_order:', problem_order_count)

# Challenge 2: Group flagged orders by cleaned supplier name.
problem_orders_by_supplier = collections.Counter()
# Challenge 3: Group flagged orders by their order month (YYYY-MM).
problem_orders_by_month = collections.Counter()
for row, problems in zip(rows, all_issues):
    if problems:
        supplier = clean_supplier(row['supplier_name'])
        problem_orders_by_supplier[supplier] += 1
        month = parse_date(row['order_date']).strftime('%Y-%m')
        problem_orders_by_month[month] += 1

print('Challenge 2: problem rows by supplier:', problem_orders_by_supplier)
print('Challenge 3: top months:', problem_orders_by_month.most_common(10))


Challenge 1: rows failing validate_order: 498
Challenge 2: problem rows by supplier: Counter({'SupplierA': 123, 'SupplierE': 114, 'SupplierB': 106, 'SupplierC': 80, 'SupplierD': 75})
Challenge 3: top months: [('2022-05', 31), ('2022-03', 29), ('2023-04', 28), ('2023-12', 28), ('2023-10', 26), ('2022-06', 25), ('2022-08', 23), ('2022-02', 22), ('2023-01', 22), ('2022-10', 22)]


## Task 4: Data quality report

| Issue | Rows affected | Fixed this week? | What should happen next |
|---|---:|---|---|
| Missing product_defects | 154 | No | Investigate missingness; choose treatment in Week 7 |
| Missing return_status | 146 | No | Check whether blanks mean no return or unknown |
| Exact duplicates | 0 | Not needed | Continue monitoring |
| Cost mismatches (>1) | 154 | No | Review source values and recalculate only with evidence |
| Zero/negative costs | 1 | No | Investigate invalid cost |
| Delivery before order | 23 | No | Verify order and delivery dates |
| Actual duration mismatch | 23 | No | Reconcile with date difference |
| On-time mismatch | 54 | No | Review on-time rule and source flags |
| Supplier names changed | 1,945 | Yes | Keep canonical names |
| Warehouse names changed | 276 | Yes | Keep canonical names |
| Delivery statuses changed | 43 | Yes | Use controlled vocabulary |
| Order dates converted | 3,000 | Yes | Keep ISO dates |
| Delivery dates converted | 3,000 | Yes | Keep ISO dates |

**Before/after summary:** The audit flagged 498 orders with at least one missing or logical issue (16.6%). The most frequent single issue was missing product defects (154), tied with cost mismatches (154). Standardisation changed supplier names in 1,945 rows, warehouse names in 276, status labels in 43, and both date fields in all 3,000 rows. No records were deleted and missing values or numerical inconsistencies remain for later investigation. The issue counts overlap and should not be added together as unique orders.

**Caution:** This assessment excludes expected missing delivery fields for non-final statuses. The `validate_order` function flags non-final rows only for other applicable problems.
